In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
project_path = Path.cwd()
processed_path = project_path / "data" / "processed"

# Load processed datasets
baggage = pd.read_csv(processed_path / "baggage_clean.csv")
flight = pd.read_csv(processed_path / "flight_clean.csv")
baggage_assignment = pd.read_csv(
    processed_path / "baggage_assignment_clean.csv"
)
baggage_scan_event = pd.read_csv(
    processed_path / "baggage_scan_event_clean.csv"
)
baggage_investigation = pd.read_csv(
    processed_path / "baggage_investigation.csv"
)

print("All Phase 3 and Phase 4 datasets loaded successfully!")

All Phase 3 and Phase 4 datasets loaded successfully!


In [2]:
datasets = {
    "Baggage": baggage,
    "Flight": flight,
    "Baggage Assignment": baggage_assignment,
    "Baggage Scan Event": baggage_scan_event,
    "Baggage Investigation": baggage_investigation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Baggage: (1000, 5)
Flight: (250, 6)
Baggage Assignment: (1500, 5)
Baggage Scan Event: (9000, 9)
Baggage Investigation: (1000, 11)


In [3]:
# Build expected airport list for every baggage item

expected_airports = (
    baggage_assignment
    .merge(
        flight[["flight_id", "source", "destination"]],
        on="flight_id",
        how="left"
    )
    .groupby("bag_id")
    .apply(
        lambda group: set(group["source"]) | set(group["destination"])
    )
    .reset_index(name="expected_airports")
)

print("Expected airport reference created!")
print("Total baggage items:", len(expected_airports))

display(expected_airports.head())

Expected airport reference created!
Total baggage items: 1000


,bag_id,expected_airports
0,BAG00001,"{DEL, LHR}"
1,BAG00002,"{DXB, JFK}"
2,BAG00003,"{DXB, JFK}"
3,BAG00004,"{DEL, MAA}"
4,BAG00005,"{SIN, MAA}"


In [4]:
# Collect unique actual scan airports for each baggage item

actual_airports = (
    baggage_scan_event
    .groupby("bag_id")["airport_code"]
    .apply(set)
    .reset_index(name="actual_airports")
)

print("Actual airport reference created!")
print("Total baggage items:", len(actual_airports))

display(actual_airports.head())

Actual airport reference created!
Total baggage items: 1000


,bag_id,actual_airports
0,BAG00001,"{DEL, LHR}"
1,BAG00002,"{DXB, JFK}"
2,BAG00003,"{DXB, JFK}"
3,BAG00004,"{DEL, MAA}"
4,BAG00005,"{SIN, MAA}"


In [5]:
# Combine expected and actual airport references

airport_comparison = expected_airports.merge(
    actual_airports,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

# Identify airports not included in the expected journey
airport_comparison["unexpected_airports"] = (
    airport_comparison.apply(
        lambda row: row["actual_airports"] - row["expected_airports"],
        axis=1
    )
)

# Flag baggage with unexpected airport scans
airport_comparison["unexpected_airport_detected"] = (
    airport_comparison["unexpected_airports"].apply(bool)
)

print("Total baggage checked:", len(airport_comparison))
print(
    "Baggage with unexpected airports:",
    airport_comparison["unexpected_airport_detected"].sum()
)

print("\nDetection summary:")
display(
    airport_comparison["unexpected_airport_detected"]
    .value_counts()
)

print("\nUnexpected airport records:")
display(
    airport_comparison[
        airport_comparison["unexpected_airport_detected"]
    ].head(10)
)

Total baggage checked: 1000
Baggage with unexpected airports: 0

Detection summary:


unexpected_airport_detected
False    1000
Name: count, dtype: int64


Unexpected airport records:


,bag_id,expected_airports,actual_airports,unexpected_airports,unexpected_airport_detected


In [6]:
# Save Rule 1 detection results

rule1_output_path = processed_path / "rule1_unexpected_airport.csv"

airport_comparison.to_csv(
    rule1_output_path,
    index=False
)

print("Rule 1 results saved successfully!")
print("File:", rule1_output_path)
print("Total records:", len(airport_comparison))

Rule 1 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule1_unexpected_airport.csv
Total records: 1000


In [7]:
# Build expected flight numbers for each baggage item

expected_flights = (
    baggage_assignment
    .merge(
        flight[["flight_id", "flight_number"]],
        on="flight_id",
        how="left"
    )
    .groupby("bag_id")["flight_number"]
    .apply(set)
    .reset_index(name="expected_flights")
)

print("Expected flight reference created!")
print("Total baggage items:", len(expected_flights))

display(expected_flights.head())

Expected flight reference created!
Total baggage items: 1000


,bag_id,expected_flights
0,BAG00001,{6E5268}
1,BAG00002,{EK7887}
2,BAG00003,{LH2805}
3,BAG00004,{LH4110}
4,BAG00005,{6E811}


In [8]:
# Collect actual non-null flight numbers for each baggage item

actual_flights = (
    baggage_scan_event
    .dropna(subset=["flight_number"])
    .groupby("bag_id")["flight_number"]
    .apply(set)
    .reset_index(name="actual_flights")
)

print("Actual flight reference created!")
print("Baggage items with flight-related scans:",
      len(actual_flights))

display(actual_flights.head())

Actual flight reference created!
Baggage items with flight-related scans: 1000


,bag_id,actual_flights
0,BAG00001,{6E5268}
1,BAG00002,{EK7887}
2,BAG00003,{LH2805}
3,BAG00004,{LH4110}
4,BAG00005,{6E811}


In [9]:
# Combine expected and actual flight references

flight_comparison = expected_flights.merge(
    actual_flights,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Replace missing actual flight sets with empty sets
flight_comparison["actual_flights"] = (
    flight_comparison["actual_flights"]
    .apply(lambda x: x if isinstance(x, set) else set())
)

# Identify unexpected flight numbers
flight_comparison["unexpected_flights"] = (
    flight_comparison.apply(
        lambda row: row["actual_flights"] - row["expected_flights"],
        axis=1
    )
)

# Flag unexpected flight numbers
flight_comparison["incorrect_flight_detected"] = (
    flight_comparison["unexpected_flights"].apply(bool)
)

print("Total baggage checked:", len(flight_comparison))

print(
    "Baggage with incorrect flight numbers:",
    flight_comparison["incorrect_flight_detected"].sum()
)

print("\nDetection summary:")
display(
    flight_comparison["incorrect_flight_detected"].value_counts()
)

print("\nIncorrect flight records:")
display(
    flight_comparison[
        flight_comparison["incorrect_flight_detected"]
    ].head(10)
)

Total baggage checked: 1000
Baggage with incorrect flight numbers: 0

Detection summary:


incorrect_flight_detected
False    1000
Name: count, dtype: int64


Incorrect flight records:


,bag_id,expected_flights,actual_flights,unexpected_flights,incorrect_flight_detected


In [10]:
# Save Rule 2 detection results

rule2_output_path = processed_path / "rule2_incorrect_flight.csv"

flight_comparison.to_csv(
    rule2_output_path,
    index=False
)

print("Rule 2 results saved successfully!")
print("File:", rule2_output_path)
print("Total records:", len(flight_comparison))

Rule 2 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule2_incorrect_flight.csv
Total records: 1000


In [11]:
# Build flight reference with expected source and destination airports

flight_reference = flight[
    ["flight_number", "source", "destination"]
].copy()

print("Flight reference created!")
print("Total flights:", len(flight_reference))

display(flight_reference.head())

Flight reference created!
Total flights: 250


,flight_number,source,destination
0,BA1924,MAA,DXB
1,AI4606,MAA,DXB
2,EK3757,MAA,DXB
3,EK1779,MAA,DXB
4,BA9035,MAA,DXB


In [12]:
# Select flight-related scan events
flight_related_events = baggage_scan_event[
    baggage_scan_event["event_type"].isin(
        ["SORTED", "LOADED", "ARRIVED", "UNLOADED"]
    )
].copy()

# Join scan events with flight reference
flight_event_analysis = flight_related_events.merge(
    flight_reference,
    on="flight_number",
    how="left",
    validate="many_to_one"
)

print("Flight-related scan events:", len(flight_related_events))
print("Events joined with flight reference:", len(flight_event_analysis))

display(
    flight_event_analysis[
        [
            "bag_id",
            "event_type",
            "airport_code",
            "flight_number",
            "source",
            "destination"
        ]
    ].head(10)
)

Flight-related scan events: 5500
Events joined with flight reference: 5500


,bag_id,event_type,airport_code,flight_number,source,destination
0,BAG00001,SORTED,DEL,6E5268,DEL,LHR
1,BAG00001,LOADED,DEL,6E5268,DEL,LHR
2,BAG00001,ARRIVED,LHR,6E5268,DEL,LHR
3,BAG00001,UNLOADED,LHR,6E5268,DEL,LHR
4,BAG00002,SORTED,DXB,EK7887,DXB,JFK
5,BAG00002,LOADED,DXB,EK7887,DXB,JFK
6,BAG00002,ARRIVED,JFK,EK7887,DXB,JFK
7,BAG00002,UNLOADED,JFK,EK7887,DXB,JFK
8,BAG00003,SORTED,DXB,LH2805,DXB,JFK
9,BAG00003,LOADED,DXB,LH2805,DXB,JFK


In [13]:
# Determine the expected airport based on event type

flight_event_analysis["expected_event_airport"] = np.where(
    flight_event_analysis["event_type"].isin(["SORTED", "LOADED"]),
    flight_event_analysis["source"],
    flight_event_analysis["destination"]
)

# Compare actual and expected airports
flight_event_analysis["airport_flight_match"] = (
    flight_event_analysis["airport_code"]
    == flight_event_analysis["expected_event_airport"]
)

print("Total flight-related events:", len(flight_event_analysis))

print(
    "Matching airport-flight events:",
    flight_event_analysis["airport_flight_match"].sum()
)

print(
    "Mismatching airport-flight events:",
    (~flight_event_analysis["airport_flight_match"]).sum()
)

print("\nValidation summary:")
display(
    flight_event_analysis["airport_flight_match"].value_counts()
)

Total flight-related events: 5500
Matching airport-flight events: 5500
Mismatching airport-flight events: 0

Validation summary:


airport_flight_match
True    5500
Name: count, dtype: int64

In [14]:
# Save Rule 3 detection results

rule3_output_path = (
    processed_path / "rule3_airport_flight_association.csv"
)

flight_event_analysis.to_csv(
    rule3_output_path,
    index=False
)

print("Rule 3 results saved successfully!")
print("File:", rule3_output_path)
print("Total records:", len(flight_event_analysis))

Rule 3 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule3_airport_flight_association.csv
Total records: 5500


In [15]:
# Build expected flight-segment reference

expected_segments = (
    baggage_assignment
    .merge(
        flight[
            ["flight_id", "flight_number", "source", "destination"]
        ],
        on="flight_id",
        how="left",
        validate="many_to_one"
    )
    .sort_values(["bag_id", "segment_sequence"])
    .reset_index(drop=True)
)

print("Expected segment reference created!")
print("Total expected segments:", len(expected_segments))
print("Unique baggage items:", expected_segments["bag_id"].nunique())

display(
    expected_segments[
        [
            "bag_id",
            "segment_sequence",
            "flight_number",
            "source",
            "destination"
        ]
    ].head(10)
)

Expected segment reference created!
Total expected segments: 1500
Unique baggage items: 1000


,bag_id,segment_sequence,flight_number,source,destination
0,BAG00001,1,6E5268,DEL,LHR
1,BAG00002,1,EK7887,DXB,JFK
2,BAG00003,1,LH2805,DXB,JFK
3,BAG00004,1,LH4110,MAA,DEL
4,BAG00005,1,6E811,MAA,SIN
5,BAG00006,1,EK9395,BOM,DXB
6,BAG00007,1,LH3358,FRA,LHR
7,BAG00008,1,EK5413,MAA,DEL
8,BAG00009,1,6E1233,BOM,DXB
9,BAG00010,1,QR3532,SIN,LHR


In [16]:
# Extract flight-related scan events

actual_segment_events = baggage_scan_event[
    baggage_scan_event["event_type"].isin(
        ["SORTED", "LOADED", "ARRIVED", "UNLOADED"]
    )
].copy()

# Attach the expected segment sequence
actual_segment_events = actual_segment_events.merge(
    expected_segments[
        ["bag_id", "flight_number", "segment_sequence"]
    ],
    on=["bag_id", "flight_number"],
    how="left",
    validate="many_to_one"
)

print("Actual flight-related events:", len(actual_segment_events))

print(
    "Events matched to expected segments:",
    actual_segment_events["segment_sequence"].notna().sum()
)

print(
    "Events without expected segment:",
    actual_segment_events["segment_sequence"].isna().sum()
)

display(
    actual_segment_events[
        [
            "bag_id",
            "event_type",
            "flight_number",
            "segment_sequence",
            "airport_code"
        ]
    ].head(10)
)

Actual flight-related events: 5500
Events matched to expected segments: 5500
Events without expected segment: 0


,bag_id,event_type,flight_number,segment_sequence,airport_code
0,BAG00001,SORTED,6E5268,1,DEL
1,BAG00001,LOADED,6E5268,1,DEL
2,BAG00001,ARRIVED,6E5268,1,LHR
3,BAG00001,UNLOADED,6E5268,1,LHR
4,BAG00002,SORTED,EK7887,1,DXB
5,BAG00002,LOADED,EK7887,1,DXB
6,BAG00002,ARRIVED,EK7887,1,JFK
7,BAG00002,UNLOADED,EK7887,1,JFK
8,BAG00003,SORTED,LH2805,1,DXB
9,BAG00003,LOADED,LH2805,1,DXB


In [17]:
# Summarize observed event types for each baggage flight segment

segment_event_summary = (
    actual_segment_events
    .groupby(["bag_id", "segment_sequence"])["event_type"]
    .apply(set)
    .reset_index(name="observed_events")
)

print("Baggage flight segments analyzed:",
      len(segment_event_summary))

display(segment_event_summary.head(10))

Baggage flight segments analyzed: 1500


,bag_id,segment_sequence,observed_events
0,BAG00001,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
1,BAG00002,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
2,BAG00003,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
3,BAG00004,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
4,BAG00005,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
5,BAG00006,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
6,BAG00007,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
7,BAG00008,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
8,BAG00009,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"
9,BAG00010,1,"{ARRIVED, SORTED, UNLOADED, LOADED}"


In [18]:
# Define the required event stages for each flight segment

required_events = {
    "SORTED",
    "LOADED",
    "ARRIVED",
    "UNLOADED"
}

# Identify missing event stages for every segment

segment_event_summary["missing_events"] = (
    segment_event_summary["observed_events"]
    .apply(lambda events: required_events - events)
)

segment_event_summary["segment_events_valid"] = (
    segment_event_summary["missing_events"].apply(
        lambda missing: len(missing) == 0
    )
)

print("Total segments:", len(segment_event_summary))

print(
    "Segments with all required events:",
    segment_event_summary["segment_events_valid"].sum()
)

print(
    "Segments with missing events:",
    (~segment_event_summary["segment_events_valid"]).sum()
)

display(
    segment_event_summary[
        ~segment_event_summary["segment_events_valid"]
    ].head(10)
)

Total segments: 1500
Segments with all required events: 1000
Segments with missing events: 500


,bag_id,segment_sequence,observed_events,missing_events,segment_events_valid
501,BAG00501,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
503,BAG00502,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
505,BAG00503,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
507,BAG00504,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
509,BAG00505,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
511,BAG00506,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
513,BAG00507,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
515,BAG00508,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
517,BAG00509,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False
519,BAG00510,2,"{ARRIVED, UNLOADED, LOADED}",{SORTED},False


In [19]:
# Count missing event patterns by segment sequence

missing_event_summary = (
    segment_event_summary
    .groupby(["segment_sequence", "segment_events_valid"])
    .size()
    .reset_index(name="segment_count")
)

print("Missing event summary:")

display(missing_event_summary)

print("\nMissing event frequency:")

display(
    segment_event_summary[
        ~segment_event_summary["segment_events_valid"]
    ]["missing_events"]
    .value_counts()
)

Missing event summary:


,segment_sequence,segment_events_valid,segment_count
0,1,True,1000
1,2,False,500



Missing event frequency:


missing_events
{SORTED}    500
Name: count, dtype: int64

In [20]:
# Save flight-segment event validation results

rule4_output_path = (
    processed_path / "rule4_flight_segment_sequence.csv"
)

segment_event_summary.to_csv(
    rule4_output_path,
    index=False
)

print("Rule 4 results saved successfully!")
print("File:", rule4_output_path)
print("Total segments:", len(segment_event_summary))
print(
    "Valid segments:",
    segment_event_summary["segment_events_valid"].sum()
)
print(
    "Segments requiring investigation:",
    (~segment_event_summary["segment_events_valid"]).sum()
)

Rule 4 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule4_flight_segment_sequence.csv
Total segments: 1500
Valid segments: 1000
Segments requiring investigation: 500


In [21]:
# Build connecting journey reference

connecting_journeys = (
    expected_segments
    .pivot(
        index="bag_id",
        columns="segment_sequence",
        values=["flight_number", "source", "destination"]
    )
)

connecting_journeys.columns = [
    f"{field}_segment_{sequence}"
    for field, sequence in connecting_journeys.columns
]

connecting_journeys = connecting_journeys.dropna().reset_index()

print("Connecting journeys:", len(connecting_journeys))

display(connecting_journeys.head())

Connecting journeys: 500


,bag_id,flight_number_segment_1,flight_number_segment_2,source_segment_1,source_segment_2,destination_segment_1,destination_segment_2
0,BAG00501,AI9913,6E2160,BOM,DXB,DXB,JFK
1,BAG00502,AI4606,EK9028,MAA,DXB,DXB,LHR
2,BAG00503,QR6074,SQ506,BOM,DOH,DOH,JFK
3,BAG00504,EK2366,LH288,MAA,DOH,DOH,FRA
4,BAG00505,QR3541,6E6845,BOM,DOH,DOH,JFK


In [23]:
# Retrieve scheduled flight timings

flight_timings = flight[
    ["flight_id", "departure_time", "arrival_time"]
].copy()

# Get first-segment arrival time for each bag
first_segment = (
    expected_segments[
        expected_segments["segment_sequence"] == 1
    ][["bag_id", "flight_id"]]
    .merge(
        flight_timings[["flight_id", "arrival_time"]],
        on="flight_id",
        how="left",
        validate="many_to_one"
    )
    .rename(columns={
        "arrival_time": "first_flight_arrival"
    })
)

# Get second-segment departure time for each bag
second_segment = (
    expected_segments[
        expected_segments["segment_sequence"] == 2
    ][["bag_id", "flight_id"]]
    .merge(
        flight_timings[["flight_id", "departure_time"]],
        on="flight_id",
        how="left",
        validate="many_to_one"
    )
    .rename(columns={
        "departure_time": "second_flight_departure"
    })
)

# Combine both segments using bag_id
connecting_journeys = first_segment.merge(
    second_segment,
    on="bag_id",
    how="inner",
    validate="one_to_one"
)

# Convert timestamps to UTC
connecting_journeys["first_flight_arrival"] = pd.to_datetime(
    connecting_journeys["first_flight_arrival"], utc=True
)

connecting_journeys["second_flight_departure"] = pd.to_datetime(
    connecting_journeys["second_flight_departure"], utc=True
)

# Calculate connection gap in minutes
connecting_journeys["connection_gap_minutes"] = (
    connecting_journeys["second_flight_departure"]
    - connecting_journeys["first_flight_arrival"]
).dt.total_seconds() / 60

print("Connection times calculated!")
print("Total connecting journeys:", len(connecting_journeys))

display(connecting_journeys.head(10))

Connection times calculated!
Total connecting journeys: 500


,bag_id,flight_id_x,first_flight_arrival,flight_id_y,second_flight_departure,connection_gap_minutes
0,BAG00501,FL0091,2026-11-01 18:00:00+00:00,FL0101,2026-11-01 06:00:00+00:00,-720.0
1,BAG00502,FL0002,2026-11-02 09:00:00+00:00,FL0012,2026-11-02 15:00:00+00:00,360.0
2,BAG00503,FL0223,2026-11-03 09:00:00+00:00,FL0233,2026-11-03 15:00:00+00:00,360.0
3,BAG00504,FL0064,2026-11-04 09:00:00+00:00,FL0074,2026-11-04 15:00:00+00:00,360.0
4,BAG00505,FL0225,2026-11-05 09:00:00+00:00,FL0235,2026-11-05 15:00:00+00:00,360.0
5,BAG00506,FL0176,2026-11-06 18:00:00+00:00,FL0186,2026-11-06 06:00:00+00:00,-720.0
6,BAG00507,FL0067,2026-11-07 09:00:00+00:00,FL0077,2026-11-07 15:00:00+00:00,360.0
7,BAG00508,FL0228,2026-11-08 09:00:00+00:00,FL0238,2026-11-08 15:00:00+00:00,360.0
8,BAG00509,FL0179,2026-11-09 18:00:00+00:00,FL0189,2026-11-09 06:00:00+00:00,-720.0
9,BAG00510,FL0200,2026-11-10 18:00:00+00:00,FL0210,2026-11-10 06:00:00+00:00,-720.0


In [24]:
# Flag impossible connection chronology

connecting_journeys["impossible_connection"] = (
    connecting_journeys["connection_gap_minutes"] < 0
)

print("Total connecting journeys:", len(connecting_journeys))

print(
    "Impossible connections:",
    connecting_journeys["impossible_connection"].sum()
)

print(
    "Valid chronological connections:",
    (~connecting_journeys["impossible_connection"]).sum()
)

print("\nConnection gap distribution:")

display(
    connecting_journeys["connection_gap_minutes"]
    .value_counts()
    .sort_index()
)

print("\nSample impossible connections:")

display(
    connecting_journeys[
        connecting_journeys["impossible_connection"]
    ].head(10)
)

Total connecting journeys: 500
Impossible connections: 185
Valid chronological connections: 315

Connection gap distribution:


connection_gap_minutes
-720.0    185
 360.0    315
Name: count, dtype: int64


Sample impossible connections:


,bag_id,flight_id_x,first_flight_arrival,flight_id_y,second_flight_departure,connection_gap_minutes,impossible_connection
0,BAG00501,FL0091,2026-11-01 18:00:00+00:00,FL0101,2026-11-01 06:00:00+00:00,-720.0,True
5,BAG00506,FL0176,2026-11-06 18:00:00+00:00,FL0186,2026-11-06 06:00:00+00:00,-720.0,True
8,BAG00509,FL0179,2026-11-09 18:00:00+00:00,FL0189,2026-11-09 06:00:00+00:00,-720.0,True
9,BAG00510,FL0200,2026-11-10 18:00:00+00:00,FL0210,2026-11-10 06:00:00+00:00,-720.0,True
10,BAG00511,FL0091,2026-11-01 18:00:00+00:00,FL0101,2026-11-01 06:00:00+00:00,-720.0,True
12,BAG00513,FL0173,2026-11-03 18:00:00+00:00,FL0183,2026-11-03 06:00:00+00:00,-720.0,True
13,BAG00514,FL0174,2026-11-04 18:00:00+00:00,FL0184,2026-11-04 06:00:00+00:00,-720.0,True
20,BAG00521,FL0191,2026-11-01 18:00:00+00:00,FL0201,2026-11-01 06:00:00+00:00,-720.0,True
21,BAG00522,FL0092,2026-11-02 18:00:00+00:00,FL0102,2026-11-02 06:00:00+00:00,-720.0,True
24,BAG00525,FL0095,2026-11-05 18:00:00+00:00,FL0105,2026-11-05 06:00:00+00:00,-720.0,True


In [25]:
# Save impossible connection detection results

rule5_output_path = (
    processed_path / "rule5_impossible_connection.csv"
)

connecting_journeys.to_csv(
    rule5_output_path,
    index=False
)

print("Rule 5 results saved successfully!")
print("File:", rule5_output_path)
print("Total connecting journeys:", len(connecting_journeys))
print(
    "Impossible connections:",
    connecting_journeys["impossible_connection"].sum()
)

Rule 5 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule5_impossible_connection.csv
Total connecting journeys: 500
Impossible connections: 185


In [28]:
# Convert event timestamps into UTC datetime values

handoff_events["transfer_time"] = pd.to_datetime(
    handoff_events["transfer_time"],
    utc=True,
    errors="raise"
)

handoff_events["second_flight_loaded_time"] = pd.to_datetime(
    handoff_events["second_flight_loaded_time"],
    utc=True,
    errors="raise"
)

# Calculate handoff time difference in minutes

handoff_events["handoff_gap_minutes"] = (
    handoff_events["second_flight_loaded_time"]
    - handoff_events["transfer_time"]
).dt.total_seconds() / 60

# Detect loading before transfer

handoff_events["loaded_before_transfer"] = (
    handoff_events["handoff_gap_minutes"] < 0
)

print("Total handoff records:", len(handoff_events))

print(
    "Second flight loaded before transfer:",
    handoff_events["loaded_before_transfer"].sum()
)

print(
    "Second flight loaded after transfer:",
    (~handoff_events["loaded_before_transfer"]).sum()
)

print("\nHandoff gap distribution:")

display(
    handoff_events["handoff_gap_minutes"]
    .value_counts()
    .sort_index()
)

print("\nSample inconsistent handoffs:")

display(
    handoff_events[
        handoff_events["loaded_before_transfer"]
    ].head(10)
)

Total handoff records: 500
Second flight loaded before transfer: 185
Second flight loaded after transfer: 315

Handoff gap distribution:


handoff_gap_minutes
-810.0    185
 270.0    315
Name: count, dtype: int64


Sample inconsistent handoffs:


,bag_id,transfer_time,second_flight_loaded_time,flight_number,handoff_gap_minutes,loaded_before_transfer
0,BAG00501,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,6E2160,-810.0,True
5,BAG00506,2026-11-06 18:45:00+00:00,2026-11-06 05:15:00+00:00,LH9324,-810.0,True
8,BAG00509,2026-11-09 18:45:00+00:00,2026-11-09 05:15:00+00:00,LH4445,-810.0,True
9,BAG00510,2026-11-10 18:45:00+00:00,2026-11-10 05:15:00+00:00,AI158,-810.0,True
10,BAG00511,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,6E2160,-810.0,True
12,BAG00513,2026-11-03 18:45:00+00:00,2026-11-03 05:15:00+00:00,SQ5280,-810.0,True
13,BAG00514,2026-11-04 18:45:00+00:00,2026-11-04 05:15:00+00:00,6E1288,-810.0,True
20,BAG00521,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,BA9164,-810.0,True
21,BAG00522,2026-11-02 18:45:00+00:00,2026-11-02 05:15:00+00:00,EK7887,-810.0,True
24,BAG00525,2026-11-05 18:45:00+00:00,2026-11-05 05:15:00+00:00,6E7032,-810.0,True


In [29]:
# Save handoff validation results

rule6_output_path = (
    processed_path / "rule6_loaded_before_transfer.csv"
)

handoff_events.to_csv(
    rule6_output_path,
    index=False
)

print("Rule 6 results saved successfully!")
print("File:", rule6_output_path)
print("Total handoff records:", len(handoff_events))
print(
    "Loaded before transfer:",
    handoff_events["loaded_before_transfer"].sum()
)

Rule 6 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule6_loaded_before_transfer.csv
Total handoff records: 500
Loaded before transfer: 185


In [30]:
# Sort scan events by event_id
ordered_events = baggage_scan_event.sort_values(
    ["bag_id", "event_id"]
).copy()

# Build event sequence for every baggage item
actual_event_sequences = (
    ordered_events
    .groupby("bag_id")["event_type"]
    .apply(list)
    .reset_index(name="actual_event_sequence")
)

# Identify direct and connecting journeys
actual_event_sequences = actual_event_sequences.merge(
    baggage_assignment.groupby("bag_id")
    .size()
    .reset_index(name="number_of_segments"),
    on="bag_id",
    how="left",
    validate="one_to_one"
)

actual_event_sequences["journey_type"] = np.where(
    actual_event_sequences["number_of_segments"] == 1,
    "DIRECT",
    "CONNECTING"
)

print("Baggage journeys analyzed:", len(actual_event_sequences))

print("\nJourney type distribution:")
display(actual_event_sequences["journey_type"].value_counts())

display(actual_event_sequences.head(10))

Baggage journeys analyzed: 1000

Journey type distribution:


journey_type
DIRECT        500
CONNECTING    500
Name: count, dtype: int64

,bag_id,actual_event_sequence,number_of_segments,journey_type
0,BAG00001,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
1,BAG00002,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
2,BAG00003,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
3,BAG00004,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
4,BAG00005,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
5,BAG00006,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
6,BAG00007,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
7,BAG00008,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
8,BAG00009,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT
9,BAG00010,"[CHECK_IN, SECURITY_SCAN, SORTED, LOADED, ARRI...",1,DIRECT


In [31]:
# Define expected event patterns

expected_direct_events = [
    "CHECK_IN",
    "SECURITY_SCAN",
    "SORTED",
    "LOADED",
    "ARRIVED",
    "UNLOADED",
    "DELIVERED"
]

expected_connecting_events = [
    "CHECK_IN",
    "SECURITY_SCAN",
    "SORTED",
    "LOADED",
    "ARRIVED",
    "UNLOADED",
    "TRANSFER",
    "LOADED",
    "ARRIVED",
    "UNLOADED",
    "DELIVERED"
]

print("Expected direct journey events:")
print(expected_direct_events)

print("\nExpected connecting journey events:")
print(expected_connecting_events)

Expected direct journey events:
['CHECK_IN', 'SECURITY_SCAN', 'SORTED', 'LOADED', 'ARRIVED', 'UNLOADED', 'DELIVERED']

Expected connecting journey events:
['CHECK_IN', 'SECURITY_SCAN', 'SORTED', 'LOADED', 'ARRIVED', 'UNLOADED', 'TRANSFER', 'LOADED', 'ARRIVED', 'UNLOADED', 'DELIVERED']


In [32]:
# Assign expected event sequence based on journey type

actual_event_sequences["expected_event_sequence"] = (
    actual_event_sequences["journey_type"].apply(
        lambda journey: (
            expected_direct_events
            if journey == "DIRECT"
            else expected_connecting_events
        )
    )
)

# Compare actual and expected sequences

actual_event_sequences["event_sequence_matches"] = (
    actual_event_sequences["actual_event_sequence"]
    == actual_event_sequences["expected_event_sequence"]
)

print("Total journeys:", len(actual_event_sequences))

print(
    "Matching event sequences:",
    actual_event_sequences["event_sequence_matches"].sum()
)

print(
    "Non-matching event sequences:",
    (~actual_event_sequences["event_sequence_matches"]).sum()
)

print("\nSequence validation by journey type:")

display(
    actual_event_sequences.groupby(
        ["journey_type", "event_sequence_matches"]
    ).size().reset_index(name="bag_count")
)

Total journeys: 1000
Matching event sequences: 1000
Non-matching event sequences: 0

Sequence validation by journey type:


,journey_type,event_sequence_matches,bag_count
0,CONNECTING,True,500
1,DIRECT,True,500


In [33]:
# Save Rule 7 event sequence validation

rule7_output_path = (
    processed_path / "rule7_journey_event_validation.csv"
)

actual_event_sequences.to_csv(
    rule7_output_path,
    index=False
)

print("Rule 7 results saved successfully!")
print("File:", rule7_output_path)
print("Total journeys:", len(actual_event_sequences))
print(
    "Matching sequences:",
    actual_event_sequences["event_sequence_matches"].sum()
)
print(
    "Non-matching sequences:",
    (~actual_event_sequences["event_sequence_matches"]).sum()
)

Rule 7 results saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\rule7_journey_event_validation.csv
Total journeys: 1000
Matching sequences: 1000
Non-matching sequences: 0


In [34]:
# Load all seven rule outputs

rule1 = pd.read_csv(processed_path / "rule1_unexpected_airport.csv")
rule2 = pd.read_csv(processed_path / "rule2_incorrect_flight.csv")
rule3 = pd.read_csv(processed_path / "rule3_airport_flight_association.csv")
rule4 = pd.read_csv(processed_path / "rule4_flight_segment_sequence.csv")
rule5 = pd.read_csv(processed_path / "rule5_impossible_connection.csv")
rule6 = pd.read_csv(processed_path / "rule6_loaded_before_transfer.csv")
rule7 = pd.read_csv(processed_path / "rule7_journey_event_validation.csv")

# Inspect the dimensions and columns

rule_outputs = {
    "Rule 1": rule1,
    "Rule 2": rule2,
    "Rule 3": rule3,
    "Rule 4": rule4,
    "Rule 5": rule5,
    "Rule 6": rule6,
    "Rule 7": rule7
}

for rule_name, df in rule_outputs.items():
    print(f"\n{rule_name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())


Rule 1
Shape: (1000, 5)
Columns: ['bag_id', 'expected_airports', 'actual_airports', 'unexpected_airports', 'unexpected_airport_detected']

Rule 2
Shape: (1000, 5)
Columns: ['bag_id', 'expected_flights', 'actual_flights', 'unexpected_flights', 'incorrect_flight_detected']

Rule 3
Shape: (5500, 13)
Columns: ['event_id', 'bag_id', 'event_type', 'airport_code', 'terminal', 'flight_number', 'event_time', 'received_at', 'source_system', 'source', 'destination', 'expected_event_airport', 'airport_flight_match']

Rule 4
Shape: (1500, 5)
Columns: ['bag_id', 'segment_sequence', 'observed_events', 'missing_events', 'segment_events_valid']

Rule 5
Shape: (500, 7)
Columns: ['bag_id', 'flight_id_x', 'first_flight_arrival', 'flight_id_y', 'second_flight_departure', 'connection_gap_minutes', 'impossible_connection']

Rule 6
Shape: (500, 6)
Columns: ['bag_id', 'transfer_time', 'second_flight_loaded_time', 'flight_number', 'handoff_gap_minutes', 'loaded_before_transfer']

Rule 7
Shape: (1000, 6)
Column

In [35]:
# Aggregate Rule 3 results at baggage level

rule3_summary = (
    rule3.groupby("bag_id")
    .agg(
        total_flight_events=("event_id", "count"),
        matching_events=("airport_flight_match", "sum"),
        mismatching_events=(
            "airport_flight_match",
            lambda x: (~x).sum()
        )
    )
    .reset_index()
)

# Create baggage-level detection flag

rule3_summary["incorrect_airport_flight_detected"] = (
    rule3_summary["mismatching_events"] > 0
)

print("Rule 3 baggage-level records:", len(rule3_summary))

print(
    "Baggage with incorrect airport-flight association:",
    rule3_summary["incorrect_airport_flight_detected"].sum()
)

display(rule3_summary.head(10))

Rule 3 baggage-level records: 1000
Baggage with incorrect airport-flight association: 0


,bag_id,total_flight_events,matching_events,mismatching_events,incorrect_airport_flight_detected
0,BAG00001,4,4,0,False
1,BAG00002,4,4,0,False
2,BAG00003,4,4,0,False
3,BAG00004,4,4,0,False
4,BAG00005,4,4,0,False
5,BAG00006,4,4,0,False
6,BAG00007,4,4,0,False
7,BAG00008,4,4,0,False
8,BAG00009,4,4,0,False
9,BAG00010,4,4,0,False


In [36]:
# Aggregate Rule 4 results at baggage level

rule4_summary = (
    rule4.groupby("bag_id")
    .agg(
        total_segments=("segment_sequence", "count"),
        valid_segments=("segment_events_valid", "sum"),
        segments_requiring_investigation=(
            "segment_events_valid",
            lambda x: (~x).sum()
        )
    )
    .reset_index()
)

# Create baggage-level investigation flag

rule4_summary["segment_event_investigation_required"] = (
    rule4_summary["segments_requiring_investigation"] > 0
)

print("Rule 4 baggage-level records:", len(rule4_summary))

print(
    "Baggage requiring segment investigation:",
    rule4_summary["segment_event_investigation_required"].sum()
)

display(rule4_summary.head(10))

Rule 4 baggage-level records: 1000
Baggage requiring segment investigation: 500


,bag_id,total_segments,valid_segments,segments_requiring_investigation,segment_event_investigation_required
0,BAG00001,1,1,0,False
1,BAG00002,1,1,0,False
2,BAG00003,1,1,0,False
3,BAG00004,1,1,0,False
4,BAG00005,1,1,0,False
5,BAG00006,1,1,0,False
6,BAG00007,1,1,0,False
7,BAG00008,1,1,0,False
8,BAG00009,1,1,0,False
9,BAG00010,1,1,0,False


In [37]:
# Select relevant Rule 5 columns

rule5_summary = rule5[
    [
        "bag_id",
        "connection_gap_minutes",
        "impossible_connection"
    ]
].copy()

# Select relevant Rule 6 columns

rule6_summary = rule6[
    [
        "bag_id",
        "handoff_gap_minutes",
        "loaded_before_transfer"
    ]
].copy()

print("Rule 5 records:", len(rule5_summary))
print("Rule 6 records:", len(rule6_summary))

print("\nRule 5 impossible connections:",
      rule5_summary["impossible_connection"].sum())

print("Rule 6 loaded before transfer:",
      rule6_summary["loaded_before_transfer"].sum())

display(rule5_summary.head())
display(rule6_summary.head())

Rule 5 records: 500
Rule 6 records: 500

Rule 5 impossible connections: 185
Rule 6 loaded before transfer: 185


,bag_id,connection_gap_minutes,impossible_connection
0,BAG00501,-720.0,True
1,BAG00502,360.0,False
2,BAG00503,360.0,False
3,BAG00504,360.0,False
4,BAG00505,360.0,False


,bag_id,handoff_gap_minutes,loaded_before_transfer
0,BAG00501,-810.0,True
1,BAG00502,270.0,False
2,BAG00503,270.0,False
3,BAG00504,270.0,False
4,BAG00505,270.0,False


In [38]:
# Start with the baggage master dataset

misroute_detection = baggage[["bag_id"]].copy()

# Merge Rule 1
misroute_detection = misroute_detection.merge(
    rule1[
        ["bag_id", "unexpected_airport_detected"]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 2
misroute_detection = misroute_detection.merge(
    rule2[
        ["bag_id", "incorrect_flight_detected"]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 3
misroute_detection = misroute_detection.merge(
    rule3_summary[
        ["bag_id", "incorrect_airport_flight_detected"]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 4
misroute_detection = misroute_detection.merge(
    rule4_summary[
        ["bag_id", "segment_event_investigation_required"]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 5
misroute_detection = misroute_detection.merge(
    rule5_summary,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 6
misroute_detection = misroute_detection.merge(
    rule6_summary,
    on="bag_id",
    how="left",
    validate="one_to_one"
)

# Merge Rule 7
misroute_detection = misroute_detection.merge(
    actual_event_sequences[
        ["bag_id", "journey_type", "event_sequence_matches"]
    ],
    on="bag_id",
    how="left",
    validate="one_to_one"
)

print("Consolidated dataset created!")
print("Total records:", len(misroute_detection))
print("Total columns:", len(misroute_detection.columns))
print("Unique baggage items:", misroute_detection["bag_id"].nunique())

display(misroute_detection.head(10))

Consolidated dataset created!
Total records: 1000
Total columns: 11
Unique baggage items: 1000


,bag_id,unexpected_airport_detected,incorrect_flight_detected,incorrect_airport_flight_detected,segment_event_investigation_required,connection_gap_minutes,impossible_connection,handoff_gap_minutes,loaded_before_transfer,journey_type,event_sequence_matches
0,BAG00001,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
1,BAG00002,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
2,BAG00003,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
3,BAG00004,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
4,BAG00005,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
5,BAG00006,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
6,BAG00007,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
7,BAG00008,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
8,BAG00009,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True
9,BAG00010,False,False,False,False,NaN,NaN,NaN,NaN,DIRECT,True


In [39]:
# Fill missing detection flags with False

detection_flag_columns = [
    "unexpected_airport_detected",
    "incorrect_flight_detected",
    "incorrect_airport_flight_detected",
    "segment_event_investigation_required",
    "impossible_connection",
    "loaded_before_transfer"
]

misroute_detection[detection_flag_columns] = (
    misroute_detection[detection_flag_columns]
    .fillna(False)
    .astype(bool)
)

# Validate missing values

print("Missing values after handling detection flags:")
display(misroute_detection.isnull().sum())

print("\nJourney type distribution:")
display(misroute_detection["journey_type"].value_counts())

print("\nDetection flag summary:")

display(
    misroute_detection[detection_flag_columns].sum()
)

Missing values after handling detection flags:


bag_id                                    0
unexpected_airport_detected               0
incorrect_flight_detected                 0
incorrect_airport_flight_detected         0
segment_event_investigation_required      0
connection_gap_minutes                  500
impossible_connection                     0
handoff_gap_minutes                     500
loaded_before_transfer                    0
journey_type                              0
event_sequence_matches                    0
dtype: int64


Journey type distribution:


journey_type
DIRECT        500
CONNECTING    500
Name: count, dtype: int64


Detection flag summary:


unexpected_airport_detected               0
incorrect_flight_detected                 0
incorrect_airport_flight_detected         0
segment_event_investigation_required    500
impossible_connection                   185
loaded_before_transfer                  185
dtype: int64

In [40]:
# Create overall investigation classification

conditions = [
    (
        misroute_detection["impossible_connection"] |
        misroute_detection["loaded_before_transfer"]
    ),
    misroute_detection["segment_event_investigation_required"],
]

classifications = [
    "TIMING_INCONSISTENCY",
    "EVENT_PATTERN_REVIEW",
]

misroute_detection["investigation_status"] = np.select(
    conditions,
    classifications,
    default="NO_ISSUE_DETECTED"
)

print("Investigation classification completed!")

print("\nInvestigation status distribution:")

display(
    misroute_detection["investigation_status"]
    .value_counts()
)

print("\nStatus by journey type:")

display(
    misroute_detection.groupby(
        ["journey_type", "investigation_status"]
    ).size().reset_index(name="baggage_count")
)

Investigation classification completed!

Investigation status distribution:


investigation_status
NO_ISSUE_DETECTED       500
EVENT_PATTERN_REVIEW    315
TIMING_INCONSISTENCY    185
Name: count, dtype: int64


Status by journey type:


,journey_type,investigation_status,baggage_count
0,CONNECTING,EVENT_PATTERN_REVIEW,315
1,CONNECTING,TIMING_INCONSISTENCY,185
2,DIRECT,NO_ISSUE_DETECTED,500


In [41]:
# Save consolidated baggage investigation dataset

final_output_path = (
    processed_path / "final_baggage_misroute_investigation.csv"
)

misroute_detection.to_csv(
    final_output_path,
    index=False
)

print("Final investigation dataset saved successfully!")
print("File:", final_output_path)
print("Total records:", len(misroute_detection))
print("Unique baggage items:", misroute_detection["bag_id"].nunique())
print("Total columns:", len(misroute_detection.columns))

print("\nFinal investigation status:")
display(misroute_detection["investigation_status"].value_counts())

Final investigation dataset saved successfully!
File: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\data\processed\final_baggage_misroute_investigation.csv
Total records: 1000
Unique baggage items: 1000
Total columns: 12

Final investigation status:


investigation_status
NO_ISSUE_DETECTED       500
EVENT_PATTERN_REVIEW    315
TIMING_INCONSISTENCY    185
Name: count, dtype: int64

In [42]:
# Reload final investigation dataset

final_verified = pd.read_csv(final_output_path)

print("FINAL DATASET VERIFICATION")
print("-" * 40)

print("Original records:", len(misroute_detection))
print("Reloaded records:", len(final_verified))

print(
    "Unique baggage IDs:",
    final_verified["bag_id"].nunique()
)

print(
    "Duplicate baggage IDs:",
    final_verified["bag_id"].duplicated().sum()
)

print(
    "Column count:",
    len(final_verified.columns)
)

print("\nInvestigation status verification:")
display(final_verified["investigation_status"].value_counts())

print("\nMissing values:")
display(final_verified.isnull().sum())

print(
    "\nRow count verification:",
    len(final_verified) == len(misroute_detection)
)

print(
    "Baggage ID verification:",
    final_verified["bag_id"].equals(misroute_detection["bag_id"])
)

FINAL DATASET VERIFICATION
----------------------------------------
Original records: 1000
Reloaded records: 1000
Unique baggage IDs: 1000
Duplicate baggage IDs: 0
Column count: 12

Investigation status verification:


investigation_status
NO_ISSUE_DETECTED       500
EVENT_PATTERN_REVIEW    315
TIMING_INCONSISTENCY    185
Name: count, dtype: int64


Missing values:


bag_id                                    0
unexpected_airport_detected               0
incorrect_flight_detected                 0
incorrect_airport_flight_detected         0
segment_event_investigation_required      0
connection_gap_minutes                  500
impossible_connection                     0
handoff_gap_minutes                     500
loaded_before_transfer                    0
journey_type                              0
event_sequence_matches                    0
investigation_status                      0
dtype: int64


Row count verification: True
Baggage ID verification: True


In [2]:
import pandas as pd
from pathlib import Path

project_path = Path.cwd()
processed_path = project_path / "data" / "processed"

baggage = pd.read_csv(processed_path / "baggage_clean.csv")
flight = pd.read_csv(processed_path / "flight_clean.csv")

baggage_assignment = pd.read_csv(
    processed_path / "baggage_assignment_clean.csv"
)

baggage_scan_event = pd.read_csv(
    processed_path / "baggage_scan_event_clean.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
datasets = {
    "Baggage": baggage,
    "Flight": flight,
    "Baggage Assignment": baggage_assignment,
    "Baggage Scan Event": baggage_scan_event
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print("Columns:", df.columns.tolist())
    print("Records:", len(df))


Baggage
--------------------------------------------------
Columns: ['bag_id', 'tag_number', 'booking_reference', 'passenger_reference', 'created_at']
Records: 1000

Flight
--------------------------------------------------
Columns: ['flight_id', 'flight_number', 'source', 'destination', 'departure_time', 'arrival_time']
Records: 250

Baggage Assignment
--------------------------------------------------
Columns: ['assignment_id', 'bag_id', 'flight_id', 'segment_sequence', 'assigned_at']
Records: 1500

Baggage Scan Event
--------------------------------------------------
Columns: ['event_id', 'bag_id', 'event_type', 'airport_code', 'terminal', 'flight_number', 'event_time', 'received_at', 'source_system']
Records: 9000


In [4]:
# Check duplicate scan events

print("Duplicate event IDs:",
      baggage_scan_event["event_id"].duplicated().sum())

print("Completely duplicate scan records:",
      baggage_scan_event.duplicated().sum())

Duplicate event IDs: 0
Completely duplicate scan records: 0


In [6]:
# Check scan event chronology

scan_check = baggage_scan_event.copy()

# Convert event_time from string to datetime
scan_check["event_time"] = pd.to_datetime(
    scan_check["event_time"],
    utc=True,
    errors="coerce"
)

# Sort events by bag and event ID for this diagnostic
scan_check = scan_check.sort_values(
    ["bag_id", "event_id"]
)

# Get the previous event timestamp for each bag
scan_check["previous_event_time"] = (
    scan_check.groupby("bag_id")["event_time"].shift(1)
)

# Calculate the time difference
scan_check["time_difference"] = (
    scan_check["event_time"] -
    scan_check["previous_event_time"]
)

# Identify decreasing timestamps
out_of_order = scan_check[
    scan_check["time_difference"] < pd.Timedelta(0)
]

print("Invalid timestamps:", scan_check["event_time"].isna().sum())
print("Out-of-order scan events:", len(out_of_order))

Invalid timestamps: 0
Out-of-order scan events: 185


In [7]:
# Inspect bags with decreasing timestamps in event-ID order

print("Affected scan-event rows:", len(out_of_order))
print("Unique affected bags:", out_of_order["bag_id"].nunique())

display(
    out_of_order[
        [
            "bag_id",
            "event_id",
            "previous_event_time",
            "event_time",
            "time_difference"
        ]
    ].head(10)
)

Affected scan-event rows: 185
Unique affected bags: 185


,bag_id,event_id,previous_event_time,event_time,time_difference
3507,BAG00501,EVT003508,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-1 days +10:30:00
3562,BAG00506,EVT003563,2026-11-06 18:45:00+00:00,2026-11-06 05:15:00+00:00,-1 days +10:30:00
3595,BAG00509,EVT003596,2026-11-09 18:45:00+00:00,2026-11-09 05:15:00+00:00,-1 days +10:30:00
3606,BAG00510,EVT003607,2026-11-10 18:45:00+00:00,2026-11-10 05:15:00+00:00,-1 days +10:30:00
3617,BAG00511,EVT003618,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-1 days +10:30:00
3639,BAG00513,EVT003640,2026-11-03 18:45:00+00:00,2026-11-03 05:15:00+00:00,-1 days +10:30:00
3650,BAG00514,EVT003651,2026-11-04 18:45:00+00:00,2026-11-04 05:15:00+00:00,-1 days +10:30:00
3727,BAG00521,EVT003728,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-1 days +10:30:00
3738,BAG00522,EVT003739,2026-11-02 18:45:00+00:00,2026-11-02 05:15:00+00:00,-1 days +10:30:00
3771,BAG00525,EVT003772,2026-11-05 18:45:00+00:00,2026-11-05 05:15:00+00:00,-1 days +10:30:00


In [8]:
# Inspect all scan events for BAG00501 in actual time order

bag_sample = baggage_scan_event[
    baggage_scan_event["bag_id"] == "BAG00501"
].copy()

bag_sample["event_time"] = pd.to_datetime(
    bag_sample["event_time"],
    utc=True
)

bag_sample = bag_sample.sort_values("event_time")

display(
    bag_sample[
        [
            "event_id",
            "event_type",
            "airport_code",
            "flight_number",
            "event_time"
        ]
    ]
)

,event_id,event_type,airport_code,flight_number,event_time
3507,EVT003508,LOADED,DXB,6E2160,2026-11-01 05:15:00+00:00
3500,EVT003501,CHECK_IN,BOM,NaN,2026-11-01 11:00:00+00:00
3501,EVT003502,SECURITY_SCAN,BOM,NaN,2026-11-01 12:00:00+00:00
3508,EVT003509,ARRIVED,JFK,6E2160,2026-11-01 12:00:00+00:00
3509,EVT003510,UNLOADED,JFK,6E2160,2026-11-01 12:15:00+00:00
3502,EVT003503,SORTED,BOM,AI9913,2026-11-01 13:00:00+00:00
3510,EVT003511,DELIVERED,JFK,NaN,2026-11-01 13:00:00+00:00
3503,EVT003504,LOADED,BOM,AI9913,2026-11-01 14:15:00+00:00
3504,EVT003505,ARRIVED,DXB,AI9913,2026-11-01 18:00:00+00:00
3505,EVT003506,UNLOADED,DXB,AI9913,2026-11-01 18:15:00+00:00


In [9]:
# Compare transfer and second-flight loading times

connecting_events = baggage_scan_event[
    baggage_scan_event["bag_id"].isin(
        out_of_order["bag_id"].unique()
    )
].copy()

connecting_events["event_time"] = pd.to_datetime(
    connecting_events["event_time"],
    utc=True
)

event_summary = (
    connecting_events[
        connecting_events["event_type"].isin(
            ["TRANSFER", "LOADED"]
        )
    ]
    .sort_values(["bag_id", "event_time"])
    .groupby("bag_id")["event_type"]
    .apply(list)
)

print("Affected bags:", len(event_summary))
print("\nEvent patterns:")
display(event_summary.value_counts())

Affected bags: 185

Event patterns:


event_type
[LOADED, LOADED, TRANSFER]    185
Name: count, dtype: int64

In [10]:
# Compare event patterns for all connecting journeys

connecting_bag_ids = baggage_assignment.groupby("bag_id").filter(
    lambda group: len(group) == 2
)["bag_id"].unique()

all_connecting_events = baggage_scan_event[
    baggage_scan_event["bag_id"].isin(connecting_bag_ids)
].copy()

all_connecting_events["event_time"] = pd.to_datetime(
    all_connecting_events["event_time"],
    utc=True
)

event_patterns = (
    all_connecting_events[
        all_connecting_events["event_type"].isin(
            ["LOADED", "TRANSFER"]
        )
    ]
    .sort_values(["bag_id", "event_time"])
    .groupby("bag_id")["event_type"]
    .apply(list)
)

print("Total connecting bags:", len(event_patterns))
print("\nEvent pattern distribution:")
display(event_patterns.value_counts())

Total connecting bags: 500

Event pattern distribution:


event_type
[LOADED, TRANSFER, LOADED]    315
[LOADED, LOADED, TRANSFER]    185
Name: count, dtype: int64

In [11]:
# Validate connecting flight-segment continuity

flight_reference = flight[
    ["flight_id", "flight_number", "source", "destination"]
]

segment_check = baggage_assignment.merge(
    flight_reference,
    on="flight_id",
    how="left"
)

segment_check = segment_check.sort_values(
    ["bag_id", "segment_sequence"]
)

segment_check["next_source"] = (
    segment_check.groupby("bag_id")["source"].shift(-1)
)

segment_check["next_flight_number"] = (
    segment_check.groupby("bag_id")["flight_number"].shift(-1)
)

connecting_segments = segment_check[
    segment_check.groupby("bag_id")["bag_id"].transform("count") == 2
].copy()

connecting_segments["continuity_matches"] = (
    connecting_segments["destination"] ==
    connecting_segments["next_source"]
)

first_segments = connecting_segments[
    connecting_segments["segment_sequence"] == 1
]

print("Connecting journeys:", first_segments["bag_id"].nunique())

print("\nFlight continuity results:")
display(
    first_segments["continuity_matches"].value_counts()
)

print("\nSample connecting routes:")
display(
    first_segments[
        [
            "bag_id",
            "flight_number",
            "source",
            "destination",
            "next_flight_number",
            "next_source",
            "continuity_matches"
        ]
    ].head(10)
)

Connecting journeys: 500

Flight continuity results:


continuity_matches
True    500
Name: count, dtype: int64


Sample connecting routes:


,bag_id,flight_number,source,destination,next_flight_number,next_source,continuity_matches
500,BAG00501,AI9913,BOM,DXB,6E2160,DXB,True
502,BAG00502,AI4606,MAA,DXB,EK9028,DXB,True
504,BAG00503,QR6074,BOM,DOH,SQ506,DOH,True
506,BAG00504,EK2366,MAA,DOH,LH288,DOH,True
508,BAG00505,QR3541,BOM,DOH,6E6845,DOH,True
510,BAG00506,SQ9662,DEL,DOH,LH9324,DOH,True
512,BAG00507,6E1896,MAA,DOH,6E4471,DOH,True
514,BAG00508,QR9273,BOM,DOH,6E1871,DOH,True
516,BAG00509,EK5247,DEL,DOH,LH4445,DOH,True
518,BAG00510,AI1853,BLR,SIN,AI158,SIN,True


In [12]:
# Check duplicate flight assignments for each baggage item

duplicate_assignments = baggage_assignment[
    baggage_assignment.duplicated(
        subset=["bag_id", "flight_id"],
        keep=False
    )
].sort_values(["bag_id", "flight_id"])

print("Duplicate baggage-flight assignments:", len(duplicate_assignments))

print(
    "Affected baggage items:",
    duplicate_assignments["bag_id"].nunique()
)

display(
    duplicate_assignments[
        ["assignment_id", "bag_id", "flight_id", "segment_sequence"]
    ].head(20)
)

Duplicate baggage-flight assignments: 0
Affected baggage items: 0


,assignment_id,bag_id,flight_id,segment_sequence


In [13]:
# Check required journey milestone availability

required_events = [
    "CHECK_IN",
    "LOADED",
    "ARRIVED",
    "UNLOADED",
    "DELIVERED"
]

milestone_summary = (
    baggage_scan_event
    .groupby("bag_id")["event_type"]
    .apply(set)
)

for event in required_events:
    present = milestone_summary.apply(lambda events: event in events).sum()
    missing = len(milestone_summary) - present

    print(f"{event}: Present = {present}, Missing = {missing}")

CHECK_IN: Present = 1000, Missing = 0
LOADED: Present = 1000, Missing = 0
ARRIVED: Present = 1000, Missing = 0
UNLOADED: Present = 1000, Missing = 0
DELIVERED: Present = 1000, Missing = 0


In [14]:
# Validate chronological order of major journey milestones

milestone_order = [
    "CHECK_IN",
    "LOADED",
    "ARRIVED",
    "UNLOADED",
    "DELIVERED"
]

milestone_events = baggage_scan_event[
    baggage_scan_event["event_type"].isin(milestone_order)
].copy()

milestone_events["event_time"] = pd.to_datetime(
    milestone_events["event_time"],
    utc=True
)

milestone_events = milestone_events.sort_values(
    ["bag_id", "event_time"]
)

milestone_events["event_rank"] = (
    milestone_events["event_type"].map(
        {event: index for index, event in enumerate(milestone_order)}
    )
)

order_check = milestone_events.groupby("bag_id")["event_rank"].apply(
    lambda ranks: ranks.is_monotonic_increasing
)

print("Baggage items checked:", len(order_check))
print("\nMilestone order results:")
display(order_check.value_counts())

print(
    "\nBags with milestone order issues:",
    (~order_check).sum()
)

Baggage items checked: 1000

Milestone order results:


event_rank
True     500
False    500
Name: count, dtype: int64


Bags with milestone order issues: 500


In [15]:
# Separate milestone-order results by journey type

segment_counts = baggage_assignment.groupby("bag_id").size()

journey_type = segment_counts.map(
    lambda count: "DIRECT" if count == 1 else "CONNECTING"
)

order_results = pd.DataFrame({
    "journey_type": journey_type,
    "order_valid": order_check
})

print("Milestone order results by journey type:")

display(
    order_results.groupby(
        ["journey_type", "order_valid"]
    ).size().unstack(fill_value=0)
)

Milestone order results by journey type:


order_valid,False,True
journey_type,,
CONNECTING,500,0
DIRECT,0,500


In [16]:
# Inspect a connecting journey without the known timing inconsistency

bag_sample = baggage_scan_event[
    baggage_scan_event["bag_id"] == "BAG00502"
].copy()

bag_sample["event_time"] = pd.to_datetime(
    bag_sample["event_time"],
    utc=True
)

bag_sample = bag_sample.sort_values("event_time")

display(
    bag_sample[
        [
            "event_id",
            "event_type",
            "airport_code",
            "flight_number",
            "event_time"
        ]
    ]
)

,event_id,event_type,airport_code,flight_number,event_time
3511,EVT003512,CHECK_IN,MAA,NaN,2026-11-02 02:00:00+00:00
3512,EVT003513,SECURITY_SCAN,MAA,NaN,2026-11-02 03:00:00+00:00
3513,EVT003514,SORTED,MAA,AI4606,2026-11-02 04:00:00+00:00
3514,EVT003515,LOADED,MAA,AI4606,2026-11-02 05:15:00+00:00
3515,EVT003516,ARRIVED,DXB,AI4606,2026-11-02 09:00:00+00:00
3516,EVT003517,UNLOADED,DXB,AI4606,2026-11-02 09:15:00+00:00
3517,EVT003518,TRANSFER,DXB,NaN,2026-11-02 09:45:00+00:00
3518,EVT003519,LOADED,DXB,EK9028,2026-11-02 14:15:00+00:00
3519,EVT003520,ARRIVED,LHR,EK9028,2026-11-02 21:00:00+00:00
3520,EVT003521,UNLOADED,LHR,EK9028,2026-11-02 21:15:00+00:00
